# Task 1 – pruning and weight quantization (Qwen2.5-1.5B)
Order matters: P3/Q3 allocations are fixed (candidates + dev-set selection) **before** their test evaluation.
Approximate A100 time: baseline 10 min, uniform conditions 2–3 h, sensitivity 20–40 min, selection 1 h, adaptive 1 h.

### Setup (run every session)
Results and large artefacts go to Google Drive so a disconnected runtime loses nothing; every step is resumable.
Use an **A100** (or L4) runtime with High-RAM: *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/adilawan1/ATML-PA4.git'   # <- edit once
DRIVE = '/content/drive/MyDrive/atml-pa4'
os.makedirs(DRIVE, exist_ok=True)
os.environ['PA4_RESULTS'] = f'{DRIVE}/results'
os.environ['PA4_CKPT'] = f'{DRIVE}/checkpoints'
os.environ['PA4_DATA'] = f'{DRIVE}/atml-pa4-datasets'   # on Drive so data/local_source_revisions.json persists
if not os.path.exists('/content/atml-pa4'):
    !git clone -q $REPO_URL /content/atml-pa4
%cd /content/atml-pa4
!git pull -q
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/record_env.py | tail -1

### If the runtime disconnects or times out
Nothing is lost: every result, partial result and training checkpoint is written to Google Drive as it is produced
(atomic writes, fsynced logs). Reconnect, re-run the **Setup** cell, run the progress cell below, then re-run the
cell that was interrupted: it resumes where it stopped (MQAR runs and Task-1 conditions restart from their beginning,
everything longer resumes mid-step). Keep this tab open; Colab Pro can still disconnect idle or very long sessions.

In [ ]:
!python scripts/progress.py   # what is finished / where each step will resume

In [ ]:
!python scripts/task1.py baseline

In [ ]:
!python scripts/task1.py uniform        # P1/P2 x {25,50,75}%, 2:4 Wanda, Q1/Q2 x {8,4,3} bits

In [ ]:
!python scripts/task1.py sensitivity    # Fisher diagonal + probe pass (calibration set only)

In [ ]:
!python scripts/task1.py candidates     # P3/Q3 candidate allocations (no evaluation data used)

In [ ]:
!python scripts/task1.py select         # choose design per budget on the 32-seq C4 dev manifest

In [ ]:
!python scripts/task1.py adaptive       # final P3 (50/75%) and Q3 (INT4/INT3 budgets)

In [ ]:
!python analysis/task1_report.py